In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib numpy pandas scipy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# การทแยงเมตริกซ์เชิงควอนตัมแบบ Krylov ของ Lattice Hamiltonians

import TutorialFeedback from '@site/src/components/TutorialFeedback';





*ประมาณการเวลาใช้งาน: 20 นาทีบน Heron r2 (หมายเหตุ: นี่เป็นการประมาณเท่านั้น เวลาจริงอาจแตกต่างออกไป)*

In [1]:
from __future__ import annotations

import warnings

import numpy as np
import pandas as pd
import scipy.linalg as la
import matplotlib.pyplot as plt

from qiskit import QuantumCircuit, transpile
from qiskit.circuit import Parameter
from qiskit.circuit.library import PauliEvolutionGate
from qiskit.primitives import StatevectorEstimator
from qiskit.quantum_info import Operator, SparsePauliOp
from qiskit.synthesis import LieTrotter, SuzukiTrotter
from qiskit.transpiler import PassManager, Layout
from qiskit.transpiler.passes import CommutativeOptimization
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_ibm_runtime import QiskitRuntimeService, EstimatorV2, Batch
from qiskit_ibm_runtime.fake_provider import FakeMarrakesh

warnings.filterwarnings("ignore")


def make_heisenberg_hamiltonian(
    num_qubits: int,
    coupling: float = 1.0,
) -> SparsePauliOp:
    """Make a Heisenberg Hamiltonian for a 1D chain of qubits with nearest-neighbor interactions."""
    terms: list[tuple[str, complex]] = []

    def append_term(q0: int, q1: int, pauli: str):
        label = ["I"] * num_qubits
        label[num_qubits - 1 - q0] = pauli[0]
        label[num_qubits - 1 - q1] = pauli[1]
        terms.append(("".join(label), coupling))

    for pauli in ("XX", "YY", "ZZ"):
        for q in range(num_qubits - 1):
            append_term(q, q + 1, pauli)
    return SparsePauliOp.from_list(terms).simplify()


def _basis_state_transition_amplitude_sparse(
    hamiltonian: SparsePauliOp,
    bra_state: int,
    ket_state: int,
) -> complex:
    """Evaluate <bra_state|H|ket_state> for computational-basis states."""
    num_qubits = hamiltonian.num_qubits
    amplitude = 0.0 + 0.0j
    for pauli, coeff in zip(hamiltonian.paulis, hamiltonian.coeffs):
        new_state = ket_state
        phase = 1.0 + 0.0j
        for q in range(num_qubits):
            x = bool(pauli.x[q])
            z = bool(pauli.z[q])
            if not x and not z:
                continue
            bit = (new_state >> q) & 1
            if x and z:
                # Y|0> = i|1>, Y|1> = -i|0>
                phase *= 1j if bit == 0 else -1j
                new_state ^= 1 << q
            elif x:
                new_state ^= 1 << q
            else:
                # Z|0> = |0>, Z|1> = -|1>
                if bit:
                    phase *= -1
        if new_state == bra_state:
            amplitude += coeff * phase
    return amplitude


def basis_state_expectation_sparse(
    hamiltonian: SparsePauliOp,
    bitstring: str,
) -> complex:
    """Evaluate <bitstring|H|bitstring>."""
    state = int(bitstring, 2)
    return _basis_state_transition_amplitude_sparse(
        hamiltonian,
        bra_state=state,
        ket_state=state,
    )


def diagonalize_single_1_subspace(
    hamiltonian: SparsePauliOp,
) -> np.ndarray:
    """Diagonalize the Hamiltonian projected onto the single-excitation subspace."""
    num_qubits = hamiltonian.num_qubits

    # Integer basis states |...010...>, with the excitation at qubit k.
    basis = [1 << k for k in range(num_qubits)]

    h_single = np.empty((num_qubits, num_qubits), dtype=complex)

    for row, bra_state in enumerate(basis):
        for col, ket_state in enumerate(basis):
            h_single[row, col] = _basis_state_transition_amplitude_sparse(
                hamiltonian,
                bra_state=bra_state,
                ket_state=ket_state,
            )
    # Remove floating-point-level asymmetry.
    h_single = 0.5 * (h_single + h_single.conj().T)
    evals, _ = np.linalg.eigh(h_single)
    return np.real(evals)


def simple_transpilation(circuit: QuantumCircuit) -> QuantumCircuit:
    """Transpilation to simplify the circuit"""
    pm = PassManager(
        [
            CommutativeOptimization(),
        ]
    )
    circuit = transpile(circuit, optimization_level=3)
    circuit = pm.run(circuit)
    return circuit


def summarize_circuit(circuit: QuantumCircuit) -> dict[str, int | str]:
    """Summarize the circuit with depth, size, and 2-qubit gate information."""
    two_qubit_total = sum(
        inst.operation.num_qubits == 2 for inst in circuit.data
    )
    two_qubit_depth = circuit.depth(lambda x: x[0].num_qubits == 2)
    return {
        "depth": circuit.depth(),
        "size": circuit.size(),
        "2q gates": two_qubit_total,
        "2q depth": two_qubit_depth,
    }


def solve_thresholded_gevp(
    h_matrix: np.ndarray,
    s_matrix: np.ndarray,
    threshold: float = 1e-10,
) -> tuple[float, np.ndarray, int]:
    """Solve H c = E S c using canonical orthogonalization of S."""
    s_vals, s_vecs = la.eigh(s_matrix)

    valid = s_vals > threshold
    if not np.any(valid):
        raise ValueError(
            "All overlap eigenvalues were removed by thresholding."
        )

    keep = valid

    orthogonalizer = s_vecs[:, keep] @ np.diag(1.0 / np.sqrt(s_vals[keep]))
    h_orth = orthogonalizer.conj().T @ h_matrix @ orthogonalizer
    h_orth = 0.5 * (h_orth + h_orth.conj().T)

    eigvals, eigvecs = la.eigh(h_orth)

    coeffs = orthogonalizer @ eigvecs[:, 0]
    normalization = np.sqrt(np.real(coeffs.conj().T @ s_matrix @ coeffs))
    coeffs /= normalization

    return float(np.real(eigvals[0])), coeffs, int(np.sum(keep))

## พื้นหลัง
บทช่วยสอนนี้จะสาธิตวิธีการนำอัลกอริทึม Krylov Quantum Diagonalization (KQD) ไปใช้ภายในบริบทของ Qiskit patterns ก่อนอื่นจะได้เรียนรู้ทฤษฎีเบื้องหลังอัลกอริทึม จากนั้นจะเห็นการสาธิตการรันจริงบน QPU

ในหลากหลายสาขาวิชา เราสนใจการหาคุณสมบัติของสถานะพื้น (ground state) ของระบบควอนตัม ตัวอย่างเช่น การทำความเข้าใจธรรมชาติพื้นฐานของอนุภาคและแรง การทำนายและเข้าใจพฤติกรรมของวัสดุที่ซับซ้อน และการทำความเข้าใจอันตรกิริยาและปฏิกิริยาทางชีวเคมี เนื่องจากการเติบโตแบบเอกซ์โปเนนเชียลของ Hilbert space และความสัมพันธ์ที่เกิดขึ้นในระบบที่พัน (entangled) กัน อัลกอริทึมแบบคลาสสิกจึงต้องเผชิญความยากลำบากในการแก้ปัญหานี้สำหรับระบบควอนตัมที่มีขนาดใหญ่ขึ้นเรื่อยๆ ในด้านหนึ่งมีแนวทางที่ใช้ประโยชน์จากฮาร์ดแวร์ควอนตัมโดยเน้นที่วิธีการเชิงแปรผัน (variational quantum methods) (เช่น [variational quantum eigensolver](/tutorials/spin-chain-vqe)) เทคนิคเหล่านี้เผชิญความท้าทายกับอุปกรณ์ในปัจจุบันเนื่องจากต้องเรียกใช้ฟังก์ชันจำนวนมากในกระบวนการหาค่าที่เหมาะสม ซึ่งเพิ่มภาระทรัพยากรอย่างมากเมื่อนำเทคนิคการลดข้อผิดพลาดขั้นสูงมาใช้ ทำให้ประสิทธิภาพของมันจำกัดอยู่กับระบบขนาดเล็ก อีกด้านหนึ่งมีวิธีการควอนตัมที่รองรับข้อผิดพลาด (fault-tolerant) ซึ่งมีการรับประกันประสิทธิภาพ (เช่น [quantum phase estimation](https://arxiv.org/abs/quant-ph/0604193)) ที่ต้องการ Circuit ลึกซึ่งรันได้เฉพาะบนอุปกรณ์ fault-tolerant เท่านั้น ด้วยเหตุผลเหล่านี้ เราจึงนำเสนออัลกอริทึมควอนตัมที่อิงบนวิธีการ subspace (ตามที่อธิบายใน [review paper](https://arxiv.org/abs/2312.00178) นี้) ซึ่งก็คืออัลกอริทึม Krylov quantum diagonalization (KQD) อัลกอริทึมนี้ทำงานได้ดีในระดับขนาดใหญ่ [\[1\]](#references) บนฮาร์ดแวร์ควอนตัมในปัจจุบัน มี[การรับประกันประสิทธิภาพ](https://arxiv.org/abs/2110.07492)ที่คล้ายคลึงกับ phase estimation รองรับเทคนิคการลดข้อผิดพลาดขั้นสูง และอาจให้ผลลัพธ์ที่ไม่สามารถเข้าถึงได้ด้วยการคำนวณแบบคลาสสิก

## ข้อกำหนดเบื้องต้น
ก่อนเริ่มบทช่วยสอนนี้ ตรวจสอบให้แน่ใจว่าได้ติดตั้งสิ่งต่อไปนี้แล้ว:

- Qiskit SDK v2.0 หรือใหม่กว่า พร้อมรองรับ [visualization](https://docs.quantum.ibm.com/api/qiskit/visualization)
- Qiskit Runtime v0.22 หรือใหม่กว่า ( `pip install qiskit-ibm-runtime` )

## การตั้งค่า

In [ ]:
try:
    service = QiskitRuntimeService()
except Exception:
    QiskitRuntimeService.save_account(
        token="<api_token>", instance="<instance>", overwrite=True
    )
    service = QiskitRuntimeService()

backend = FakeMarrakesh()

## ขั้นตอนที่ 1: แปลงอินพุตแบบคลาสสิกเป็นปัญหาควอนตัม
### Krylov space
Krylov space $\mathcal{K}^r$ ลำดับที่ $r$ คือปริภูมิที่ถูก span โดยเวกเตอร์ที่ได้จากการคูณกำลังสูงกว่าของเมตริกซ์ $A$ ไปจนถึง $r-1$ กับเวกเตอร์อ้างอิง $\vert v \rangle$

$$
\mathcal{K}^r = \left{ \vert v \rangle, A \vert v \rangle, A^2 \vert v \rangle, ..., A^{r-1} \vert v \rangle \right}
$$

ถ้าเมตริกซ์ $A$ คือ Hamiltonian $H$ เราจะเรียกปริภูมิที่สอดคล้องกันว่า power Krylov space $\mathcal{K}_P$ ในกรณีที่ $A$ คือตัวดำเนินการวิวัฒนาการตามเวลาที่สร้างขึ้นโดย Hamiltonian $U=e^{-iHt}$ เราจะเรียกปริภูมินั้นว่า unitary Krylov space $\mathcal{K}_U$ power Krylov subspace ที่เราใช้ในการคำนวณแบบคลาสสิกไม่สามารถสร้างขึ้นโดยตรงบนคอมพิวเตอร์ควอนตัม เนื่องจาก $H$ ไม่ใช่ตัวดำเนินการยูนิทารี แต่เราสามารถใช้ตัวดำเนินการวิวัฒนาการตามเวลา $U = e^{-iHt}$ ซึ่งสามารถแสดงให้เห็นว่าให้[การรับประกันการลู่เข้า](https://arxiv.org/abs/2110.07492)ที่คล้ายคลึงกับ power method กำลังของ $U$ จึงกลายเป็นขั้นเวลาต่างกัน $U^k = e^{-iH(kt)}$

$$
\mathcal{K}_U^r = \left{ \vert \psi \rangle, U \vert \psi \rangle, U^2 \vert \psi \rangle, ..., U^{r-1} \vert \psi \rangle \right}
$$

ดูภาคผนวกสำหรับการอนุมานโดยละเอียดว่า unitary Krylov space ช่วยให้แสดง eigenstates พลังงานต่ำได้อย่างแม่นยำได้อย่างไร

### อัลกอริทึม Krylov quantum diagonalization
เมื่อกำหนด Hamiltonian $H$ ที่เราต้องการทแยงเมตริกซ์ อันดับแรกเราพิจารณา unitary Krylov space $\mathcal{K}_U$ ที่สอดคล้องกัน เป้าหมายคือการหาการแทนค่าแบบกระชับของ Hamiltonian ใน $\mathcal{K}_U$ ซึ่งเราจะเรียกว่า $\tilde{H}$ องค์ประกอบเมตริกซ์ของ $\tilde{H}$ ซึ่งเป็น projection ของ Hamiltonian ลงใน Krylov space สามารถคำนวณได้โดยการคำนวณค่าความคาดหวังต่อไปนี้

$$
\tilde{H}_{mn} = \langle \psi_m \vert H \vert \psi_n \rangle =
$$
$$
= \langle \psi \vert  e^{i H t_m}   H e^{-i H t_n} \vert \psi \rangle
$$
$$
= \langle \psi \vert  e^{i H m dt}   H e^{-i H n dt} \vert \psi \rangle
$$

โดยที่ $\vert \psi_n \rangle = e^{-i H t_n} \vert \psi \rangle$ คือเวกเตอร์ของ unitary Krylov space และ $t_n = n dt$ คือผลคูณของขั้นเวลา $dt$ ที่เลือก บนคอมพิวเตอร์ควอนตัม การคำนวณองค์ประกอบเมตริกซ์แต่ละตัวสามารถทำได้ด้วยอัลกอริทึมใดก็ตามที่ช่วยให้ได้ค่า overlap ระหว่างสถานะควอนตัม บทช่วยสอนนี้เน้นที่ Hadamard test เนื่องจาก $\mathcal{K}_U$ มีมิติ $r$ Hamiltonian ที่ projected ลงใน subspace จะมีมิติ $r \times r$ ด้วยค่า $r$ ที่เล็กพอ (โดยทั่วไป $r<<100$ เพียงพอที่จะได้การลู่เข้าของการประมาณค่า eigenenergies) เราสามารถทแยงเมตริกซ์ projected Hamiltonian $\tilde{H}$ ได้อย่างง่ายดาย อย่างไรก็ตาม เราไม่สามารถทแยงเมตริกซ์ $\tilde{H}$ โดยตรงได้เนื่องจากความไม่ตั้งฉากกันของเวกเตอร์ Krylov space เราต้องวัดค่า overlap และสร้างเมตริกซ์ $\tilde{S}$

$$
\tilde{S}_{mn} = \langle \psi_m \vert \psi_n \rangle
$$

สิ่งนี้ช่วยให้เราแก้ปัญหา eigenvalue ในปริภูมิที่ไม่ตั้งฉาก (เรียกอีกอย่างว่า generalized eigenvalue problem)

$$
\tilde{H} \ \vec{c} = E \ \tilde{S} \ \vec{c}
$$

จากนั้นเราสามารถหาค่าประมาณของ eigenvalues และ eigenstates ของ $H$ ได้โดยดูที่ค่าต่างๆ ของ $\tilde{H}$ ตัวอย่างเช่น ค่าประมาณพลังงานสถานะพื้นได้มาจากการนำ eigenvalue ที่เล็กที่สุด $c$ และสถานะพื้นจาก eigenvector $\vec{c}$ ที่สอดคล้องกัน สัมประสิทธิ์ใน $\vec{c}$ กำหนดการมีส่วนร่วมของเวกเตอร์ต่างๆ ที่ span $\mathcal{K}_U$

![fig1.png](../docs/images/tutorials/krylov-subspace-diagonalization/fc662b76-8ad7-4a6c-8c49-5f08c125aee8.avif)

ภาพแสดง Circuit representation ของ modified Hadamard test ซึ่งเป็นวิธีการที่ใช้คำนวณค่า overlap ระหว่างสถานะควอนตัมต่างๆ สำหรับองค์ประกอบเมตริกซ์แต่ละตัว $\tilde{H}_{i,j}$ จะมีการทำ Hadamard test ระหว่างสถานะ $\vert \psi_i \rangle$ และ $\vert \psi_j \rangle$ ซึ่งเน้นให้เห็นในภาพด้วยรูปแบบสีของส่วนประกอบเมตริกซ์และการดำเนินการ $\text{Prep} \; \psi_i$ และ $\text{Prep} \; \psi_j$ ที่สอดคล้องกัน ดังนั้นจึงต้องมีชุด Hadamard tests สำหรับทุกคู่ที่เป็นไปได้ของเวกเตอร์ Krylov space เพื่อคำนวณองค์ประกอบเมตริกซ์ทั้งหมดของ projected Hamiltonian $\tilde{H}$ สายบนใน Hadamard test circuit คือ ancilla Qubit ที่ถูกวัดในฐาน X หรือ Y ค่าความคาดหวังของมันกำหนดค่า overlap ระหว่างสถานะต่างๆ สายล่างแทน Qubit ทั้งหมดของระบบ Hamiltonian การดำเนินการ $\text{Prep} \; \psi_i$ เตรียม system qubit ในสถานะ $\vert \psi_i \rangle$ โดยควบคุมด้วยสถานะของ ancilla qubit (เช่นเดียวกันสำหรับ $\text{Prep} \; \psi_j$) และการดำเนินการ $P$ แทน Pauli decomposition ของระบบ Hamiltonian $H = \sum_i P_i$ การอนุมานโดยละเอียดเพิ่มเติมของการดำเนินการที่คำนวณโดย Hadamard test ให้ไว้ด้านล่าง

#### กำหนด Hamiltonian
มาพิจารณา Heisenberg Hamiltonian สำหรับ $N$ Qubits บน linear chain: $H= \sum_{i,j}^N X_i X_j + Y_i Y_j - J Z_i Z_j$

In [3]:
# Problem definition for the simulator example
num_qubits = 12
hamiltonian = make_heisenberg_hamiltonian(num_qubits=num_qubits, coupling=1.0)
ref_bitstring = "000001000000"
ref_energy = basis_state_expectation_sparse(hamiltonian, ref_bitstring)

print("Hamiltonian:")
print(hamiltonian)
print(f"Reference state: |{ref_bitstring}>")
print("Reference energy: ", ref_energy)

Hamiltonian:
SparsePauliOp(['IIIIIIIIIIXX', 'IIIIIIIIIXXI', 'IIIIIIIIXXII', 'IIIIIIIXXIII', 'IIIIIIXXIIII', 'IIIIIXXIIIII', 'IIIIXXIIIIII', 'IIIXXIIIIIII', 'IIXXIIIIIIII', 'IXXIIIIIIIII', 'XXIIIIIIIIII', 'IIIIIIIIIIYY', 'IIIIIIIIIYYI', 'IIIIIIIIYYII', 'IIIIIIIYYIII', 'IIIIIIYYIIII', 'IIIIIYYIIIII', 'IIIIYYIIIIII', 'IIIYYIIIIIII', 'IIYYIIIIIIII', 'IYYIIIIIIIII', 'YYIIIIIIIIII', 'IIIIIIIIIIZZ', 'IIIIIIIIIZZI', 'IIIIIIIIZZII', 'IIIIIIIZZIII', 'IIIIIIZZIIII', 'IIIIIZZIIIII', 'IIIIZZIIIIII', 'IIIZZIIIIIII', 'IIZZIIIIIIII', 'IZZIIIIIIIII', 'ZZIIIIIIIIII'],
              coeffs=[1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j,
 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j,
 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j,
 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j])
Reference state: |000001000000>
Reference energy:  (7+0j)


#### Set the parameters for the algorithm

Based on the upper-bounds on the Hamiltonian norm, Ref. [\[1\]](#references) heuristically suggests the time step $\Delta t$ as $\pi/\|H\|$.
Because the spectral norm $\|H\|$ is hard to compute, we instead use its upper bound:

$$
\begin{equation*}
\|H\| \le \sum_{i=0}^{n-2}\underbrace{\|X_i X_{i+1}+Y_{i} Y_{i+1}+Z_{i} Z_{i+1}\|}_{4 \times 4 \text{ matrix, easy to calculate the norm}} = 3(n-1).
\end{equation*}
$$

We set the Krylov dimension to $r=10$ and the number of Trotter steps per time step to $5$: a large enough Krylov space to resolve the low-lying spectrum while keeping the deepest circuit ($t_{\max}=(r-1)\Delta t$) affordable, and enough Trotter steps to keep the discretization error small at that deepest circuit.

In [4]:
dt = np.pi / (3 * (num_qubits - 1))
print("dt in Krylov basis:  ", dt)

krylov_dim = 10
num_trotter_steps = 5

dt in Krylov basis:   0.09519977738150888


#### กำหนดพารามิเตอร์สำหรับอัลกอริทึม
เราเลือกค่า `dt` สำหรับขั้นเวลาด้วยวิธีการฮิวริสติก (อิงจากขอบเขตบนของ norm ของ Hamiltonian) Ref [\[2\]](#references) แสดงว่าขั้นเวลาที่เล็กพอคือ $\pi/\vert \vert H \vert \vert$ และว่าการประมาณค่าต่ำกว่าจริงนั้นดีกว่าการประมาณค่าสูงกว่าจริงในระดับหนึ่ง เนื่องจากการประมาณสูงเกินไปอาจทำให้มีส่วนร่วมจากสถานะพลังงานสูงมาทำลายแม้แต่สถานะที่เหมาะสมที่สุดใน Krylov space ในทางกลับกัน การเลือก $dt$ ที่เล็กเกินไปทำให้ conditioning ของ Krylov subspace แย่ลง เนื่องจากเวกเตอร์ฐาน Krylov แตกต่างกันน้อยลงจากขั้นเวลาหนึ่งไปสู่อีกขั้นเวลาหนึ่ง

In [5]:
qc_ref = QuantumCircuit(num_qubits)
for i, b in enumerate(reversed(ref_bitstring)):
    if b == "1":
        qc_ref.x(i)
display(qc_ref.draw("mpl", scale=0.5))

<Image src="../docs/images/tutorials/krylov-quantum-diagonalization/extracted-outputs/fb2879c5-0.avif" alt="Output of the previous code cell" />

#### Time evolution

Realize the time-evolution operator generated by the Hamiltonian, approximated by simple [Lie-Trotterization](/docs/api/qiskit/qiskit.synthesis.LieTrotter).

In [6]:
t = Parameter("t")

evol_gate = PauliEvolutionGate(
    hamiltonian,
    time=t,
    synthesis=LieTrotter(reps=num_trotter_steps),
    label="U(t)",
)

# Synthesize U(t) first and then control the synthesized circuit.
# This makes the controlled structure visible in the circuit drawer.
evolution_circuit = QuantumCircuit(num_qubits, name="U(t)")
evolution_circuit.append(evol_gate, range(num_qubits))
evolution_circuit = simple_transpilation(evolution_circuit)

# Make a controlled version of the evolution circuit.
controlled_evolution_gate = evolution_circuit.to_gate(label="U(t)").control(
    1, label="C-U(t)"
)
display(
    evolution_circuit.assign_parameters({t: 1.5}).draw(
        "mpl", scale=0.5, fold=-1
    )
)

<Image src="../docs/images/tutorials/krylov-quantum-diagonalization/extracted-outputs/017b8548-0.avif" alt="Output of the previous code cell" />

และกำหนดพารามิเตอร์อื่นๆ ของอัลกอริทึม เพื่อวัตถุประสงค์ของบทช่วยสอนนี้ เราจะจำกัดตัวเองให้ใช้ Krylov space ที่มีเพียงห้ามิติ ซึ่งค่อนข้างจำกัด

In [7]:
ancilla = 0
system_qubits = list(range(1, num_qubits + 1))

extended_swap_test = QuantumCircuit(num_qubits + 1)

# Append state preparation part
extended_swap_test = extended_swap_test.compose(qc_ref, system_qubits)

# Prepare the coherent branch label, (|0> + |1>) / sqrt(2).
extended_swap_test.h(ancilla)

# Apply U(t) only to the |1> branch of the ancilla.
extended_swap_test.append(
    controlled_evolution_gate, [ancilla] + system_qubits
)

# Decompose once more for visualization so that control bullets are visible.
display(extended_swap_test.draw("mpl", fold=-1))

<Image src="../docs/images/tutorials/krylov-quantum-diagonalization/extracted-outputs/1260f66a-0.avif" alt="Output of the previous code cell" />

#### การเตรียมสถานะ
เลือกสถานะอ้างอิง $\vert \psi \rangle$ ที่มีค่า overlap บางส่วนกับสถานะพื้น สำหรับ Hamiltonian นี้ เราใช้สถานะที่มีการกระตุ้นที่ Qubit กลาง $\vert 00..010...00 \rangle$ เป็นสถานะอ้างอิงของเรา

In [8]:
n_qubits = hamiltonian.num_qubits

observable_labels = [
    "Re S_0d",
    "Im S_0d",
    "Re H_0d",
    "Im H_0d",
]

# X ⊗ I and Y ⊗ I.
# Qiskit's Pauli-label convention places qubit 0 on the rightmost character,
# so the ancilla Pauli is appended to the right.
obs_x_identity = SparsePauliOp("I" * n_qubits + "X")
obs_y_identity = SparsePauliOp("I" * n_qubits + "Y")

# X ⊗ H and Y ⊗ H.
obs_x_hamiltonian = SparsePauliOp.from_list(
    [
        (label + "X", coeff)
        for label, coeff in zip(
            hamiltonian.paulis.to_labels(),
            hamiltonian.coeffs,
        )
    ]
)

obs_y_hamiltonian = SparsePauliOp.from_list(
    [
        (label + "Y", coeff)
        for label, coeff in zip(
            hamiltonian.paulis.to_labels(),
            hamiltonian.coeffs,
        )
    ]
)

observables = [
    obs_x_identity,
    obs_y_identity,
    obs_x_hamiltonian,
    obs_y_hamiltonian,
]

for obs, label in zip(observables, observable_labels):
    print(f"Observable: {label}")
    print(obs)
    print()

Observable: Re S_0d
SparsePauliOp(['IIIIIIIIIIIIX'],
              coeffs=[1.+0.j])

Observable: Im S_0d
SparsePauliOp(['IIIIIIIIIIIIY'],
              coeffs=[1.+0.j])

Observable: Re H_0d
SparsePauliOp(['IIIIIIIIIIXXX', 'IIIIIIIIIXXIX', 'IIIIIIIIXXIIX', 'IIIIIIIXXIIIX', 'IIIIIIXXIIIIX', 'IIIIIXXIIIIIX', 'IIIIXXIIIIIIX', 'IIIXXIIIIIIIX', 'IIXXIIIIIIIIX', 'IXXIIIIIIIIIX', 'XXIIIIIIIIIIX', 'IIIIIIIIIIYYX', 'IIIIIIIIIYYIX', 'IIIIIIIIYYIIX', 'IIIIIIIYYIIIX', 'IIIIIIYYIIIIX', 'IIIIIYYIIIIIX', 'IIIIYYIIIIIIX', 'IIIYYIIIIIIIX', 'IIYYIIIIIIIIX', 'IYYIIIIIIIIIX', 'YYIIIIIIIIIIX', 'IIIIIIIIIIZZX', 'IIIIIIIIIZZIX', 'IIIIIIIIZZIIX', 'IIIIIIIZZIIIX', 'IIIIIIZZIIIIX', 'IIIIIZZIIIIIX', 'IIIIZZIIIIIIX', 'IIIZZIIIIIIIX', 'IIZZIIIIIIIIX', 'IZZIIIIIIIIIX', 'ZZIIIIIIIIIIX'],
              coeffs=[1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j,
 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j,
 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j, 1

![Output of the previous code cell](../docs/images/tutorials/krylov-quantum-diagonalization/extracted-outputs/70161afe-8ace-4642-894a-cd21ed77a3b9-0.avif)

#### วิวัฒนาการตามเวลา
เราสามารถทำให้ตัวดำเนินการวิวัฒนาการตามเวลาที่สร้างโดย Hamiltonian ที่กำหนด $U=e^{-iHt}$ เป็นจริงได้ผ่านทาง [Lie-Trotter approximation](https://docs.quantum.ibm.com/api/qiskit/qiskit.synthesis.LieTrotter)

In [9]:
def make_reduced_heisenberg_observables(
    ref_bitstring: str,
    coupling: float = 1.0,
) -> tuple[SparsePauliOp, SparsePauliOp, float]:
    """Build X⊗(H-T), Y⊗(H-T), and tau."""
    n_qubits = len(ref_bitstring)

    shifted_terms: list[tuple[str, complex]] = []
    tau = 0.0

    def bit(q: int) -> str:
        return ref_bitstring[n_qubits - 1 - q]

    def append_term(q0: int, q1: int, pauli: str):
        label = ["I"] * n_qubits
        label[n_qubits - 1 - q0] = pauli[0]
        label[n_qubits - 1 - q1] = pauli[1]
        shifted_terms.append(("".join(label), coupling))

    for q in range(n_qubits - 1):
        same_occupation = bit(q) == bit(q + 1)

        # ZZ contribution to tau
        tau += coupling * (1.0 if same_occupation else -1.0)

        # XX + YY survives only for opposite occupations.
        if not same_occupation:
            append_term(q, q + 1, "XX")
            append_term(q, q + 1, "YY")

    if shifted_terms:
        obs_x_shifted_hamiltonian = SparsePauliOp.from_list(
            [(label + "X", coeff) for label, coeff in shifted_terms]
        )
        obs_y_shifted_hamiltonian = SparsePauliOp.from_list(
            [(label + "Y", coeff) for label, coeff in shifted_terms]
        )
    else:
        obs_x_shifted_hamiltonian = SparsePauliOp(
            "I" * n_qubits + "X", coeffs=[0.0]
        )
        obs_y_shifted_hamiltonian = SparsePauliOp(
            "I" * n_qubits + "Y", coeffs=[0.0]
        )

    return obs_x_shifted_hamiltonian, obs_y_shifted_hamiltonian, tau


obs_x_shifted_hamiltonian, obs_y_shifted_hamiltonian, shift_tau = (
    make_reduced_heisenberg_observables(ref_bitstring)
)

print("Observable: Re shifted H_0d")
print(obs_x_shifted_hamiltonian)
print()

print("Observable: Im shifted H_0d")
print(obs_y_shifted_hamiltonian)
print()

print("tau =", shift_tau)

observables = [
    obs_x_identity,
    obs_y_identity,
    obs_x_shifted_hamiltonian,
    obs_y_shifted_hamiltonian,
]
observable_labels = [
    "Re S_0d",
    "Im S_0d",
    "Re shifted H_0d",
    "Im shifted H_0d",
]

Observable: Re shifted H_0d
SparsePauliOp(['IIIIIXXIIIIIX', 'IIIIIYYIIIIIX', 'IIIIXXIIIIIIX', 'IIIIYYIIIIIIX'],
              coeffs=[1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j])

Observable: Im shifted H_0d
SparsePauliOp(['IIIIIXXIIIIIY', 'IIIIIYYIIIIIY', 'IIIIXXIIIIIIY', 'IIIIYYIIIIIIY'],
              coeffs=[1.+0.j, 1.+0.j, 1.+0.j, 1.+0.j])

tau = 7.0


### Step 2: Optimize problem for quantum hardware execution

Now we turn the abstract extended-swap-test circuit into a hardware-oriented template.
Before that, we further optimize the circuit in the abstract level first.

#### Compare Hamiltonian term ordering

First, we compare different orderings of the Pauli terms in the Heisenberg Hamiltonian for the Hamiltonian simulation.
The Hamiltonian itself is unchanged, but the ordering affects how the product-formula circuit is generated and how much the structure can be parallelized in the circuit.
For example, the naive ordering lists all nearest-neighbor $XX$ terms, then all $YY$ terms, then all $ZZ$ terms.
This places adjacent edges such as $(0,1)$ and $(1,2)$ next to each other, so they cannot be executed in parallel.
The even-then-odd ordering visits disjoint even edges first, followed by odd edges, which exposes parallel two-qubit layers.
The even-odd edge-grouped ordering goes one step further: for each edge, it keeps the local $XX$, $YY$, and $ZZ$ terms together, while still visiting even edges before odd edges.
We expect the even-then-odd and even-odd edge-grouped orderings to reduce circuit depth by exposing parallel two-qubit layers, and the edge-grouped ordering to additionally reduce the Trotter error because the local two-qubit interaction on the same edge is handled as a compact block.

![Pauli evolutions with different term ordering](../docs/images/tutorials/krylov-quantum-diagonalization/ham_ordering.avif)

The Hamiltonian ordering also affects the Trotter error.
If we place noncommuting terms next to each other, the basis transitions occur more often, which induces more Trotter error.
By grouping terms that require the same Pauli-basis transformation, redundant basis-changes can be avoided.

Here, the comparison uses the largest evolution time that appears in the first-row Krylov estimates,
$t_{\max} = (r-1)\Delta t,$
using the same transpilation condition.

In order to measure the Trotter error, we use **process infidelity** between the Trotterized circuit and the exact Hamiltonian evolution,
$$
\begin{equation*}
\text{Infidelity}(U,V)
=
1-F(U,V)
=
1 - \frac{|\operatorname{Tr}(U^\dagger V)|^2}{d^2},
\end{equation*}
$$
where $d=2^n$ is the dimension of the Hilbert space.

This diagnostic uses dense matrices, so it is suitable for this small 12-qubit example but not intended as a scalable subroutine.

In [10]:
# The comparison uses the largest time that appears in the first-row Krylov estimates.
# Circuit depth does not depend on this numeric value, but the Trotter error does.
comparison_time = (krylov_dim - 1) * dt


def make_heisenberg_hamiltonian_ordered(
    num_qubits: int,
    ordering: str,
    coupling: float = 1.0,
) -> SparsePauliOp:
    """Return the same Heisenberg Hamiltonian with a specified term ordering."""
    terms: list[tuple[str, complex]] = []
    even_edges = [(q, q + 1) for q in range(0, num_qubits - 1, 2)]
    odd_edges = [(q, q + 1) for q in range(1, num_qubits - 1, 2)]

    def append_term(q0: int, q1: int, pauli: str):
        label = ["I"] * num_qubits
        label[num_qubits - 1 - q0] = pauli[0]
        label[num_qubits - 1 - q1] = pauli[1]
        terms.append(("".join(label), coupling))

    if ordering == "naive":
        for pauli in ("XX", "YY", "ZZ"):
            for q in range(num_qubits - 1):
                append_term(q, q + 1, pauli)
    elif ordering == "even-then-odd":
        for pauli in ("XX", "YY", "ZZ"):
            for q0, q1 in even_edges + odd_edges:
                append_term(q0, q1, pauli)
    elif ordering == "even-odd edge-grouped":
        for q0, q1 in even_edges + odd_edges:
            for pauli in ("XX", "YY", "ZZ"):
                append_term(q0, q1, pauli)
    else:
        raise ValueError(f"Unknown ordering: {ordering}")

    return SparsePauliOp.from_list(terms).simplify()


def build_numeric_evolution_circuit(
    hamiltonian: SparsePauliOp,
    synthesis,
    time_value: float,
    **synthesis_kwargs,
) -> QuantumCircuit:
    """Build a numeric circuit for exp(-i H t) with a chosen synthesis rule."""
    evolution_gate = PauliEvolutionGate(
        hamiltonian,
        time=time_value,
        synthesis=synthesis(**synthesis_kwargs),
    )
    circuit = QuantumCircuit(hamiltonian.num_qubits)
    circuit.append(evolution_gate, range(hamiltonian.num_qubits))
    return circuit


def process_infidelity(
    circuit: QuantumCircuit,
    exact_matrix: np.ndarray,
) -> float:
    """Return 1 - |Tr(U_circuit† U_exact) / d|²."""
    circuit_matrix = np.asarray(Operator(circuit).data)
    dim = circuit_matrix.shape[0]
    normalized_trace = np.vdot(circuit_matrix, exact_matrix) / dim
    fidelity = np.abs(normalized_trace) ** 2
    return float(np.clip(1.0 - fidelity, 0.0, 1.0))


hamiltonians_by_ordering = {
    ordering: make_heisenberg_hamiltonian_ordered(
        num_qubits, ordering, coupling=1.0
    )
    for ordering in ["naive", "even-then-odd", "even-odd edge-grouped"]
}

print(f"Comparison time: {comparison_time}\n")

for order_name, ham_ordered in hamiltonians_by_ordering.items():
    print(f"{order_name}:")
    print([op for op, _ in ham_ordered.to_list()])
    print()

print("Precomputing the exact evolution operator... ", end="")
exact_matrix = la.expm(-1j * comparison_time * hamiltonian.to_matrix())
print("Done")

Comparison time: 0.8567979964335799

naive:
['IIIIIIIIIIXX', 'IIIIIIIIIXXI', 'IIIIIIIIXXII', 'IIIIIIIXXIII', 'IIIIIIXXIIII', 'IIIIIXXIIIII', 'IIIIXXIIIIII', 'IIIXXIIIIIII', 'IIXXIIIIIIII', 'IXXIIIIIIIII', 'XXIIIIIIIIII', 'IIIIIIIIIIYY', 'IIIIIIIIIYYI', 'IIIIIIIIYYII', 'IIIIIIIYYIII', 'IIIIIIYYIIII', 'IIIIIYYIIIII', 'IIIIYYIIIIII', 'IIIYYIIIIIII', 'IIYYIIIIIIII', 'IYYIIIIIIIII', 'YYIIIIIIIIII', 'IIIIIIIIIIZZ', 'IIIIIIIIIZZI', 'IIIIIIIIZZII', 'IIIIIIIZZIII', 'IIIIIIZZIIII', 'IIIIIZZIIIII', 'IIIIZZIIIIII', 'IIIZZIIIIIII', 'IIZZIIIIIIII', 'IZZIIIIIIIII', 'ZZIIIIIIIIII']

even-then-odd:
['IIIIIIIIIIXX', 'IIIIIIIIXXII', 'IIIIIIXXIIII', 'IIIIXXIIIIII', 'IIXXIIIIIIII', 'XXIIIIIIIIII', 'IIIIIIIIIXXI', 'IIIIIIIXXIII', 'IIIIIXXIIIII', 'IIIXXIIIIIII', 'IXXIIIIIIIII', 'IIIIIIIIIIYY', 'IIIIIIIIYYII', 'IIIIIIYYIIII', 'IIIIYYIIIIII', 'IIYYIIIIIIII', 'YYIIIIIIIIII', 'IIIIIIIIIYYI', 'IIIIIIIYYIII', 'IIIIIYYIIIII', 'IIIYYIIIIIII', 'IYYIIIIIIIII', 'IIIIIIIIIIZZ', 'IIIIIIIIZZII', 'IIIIIIZZIIII', 'IIIIZZIII

#### Hadamard test
![fig2.png](../docs/images/tutorials/krylov-subspace-diagonalization/c5263851-6067-4ca2-8e0c-a835631cdc7f.avif)

$$
\begin{equation*}
    |0\rangle|0\rangle^N \quad\longrightarrow\quad \frac{1}{\sqrt{2}}\Big(|0\rangle + |1\rangle \Big)|0\rangle^N \quad\longrightarrow\quad \frac{1}{\sqrt{2}}\Big(|0\rangle|0\rangle^N+|1\rangle |\psi_i\rangle\Big) \quad\longrightarrow\quad \frac{1}{\sqrt{2}}\Big(|0\rangle |0\rangle^N+|1\rangle P |\psi_i\rangle\Big) \quad\longrightarrow\quad\frac{1}{\sqrt{2}}\Big(|0\rangle |\psi_j\rangle+|1\rangle P|\psi_i\rangle\Big)
\end{equation*}
$$

โดยที่ $P$ คือหนึ่งในพจน์ใน decomposition ของ Hamiltonian $H=\sum P$ และ $\text{Prep} \; \psi_i$, $\text{Prep} \; \psi_j$ คือการดำเนินการแบบ controlled ที่เตรียม $|\psi_i\rangle$, $|\psi_j\rangle$ เวกเตอร์ของ unitary Krylov space โดยที่ $|\psi_k\rangle = e^{-i H k dt } \vert \psi \rangle = e^{-i H k dt } U_{\psi} \vert 0 \rangle^N$ ในการวัด $X$ ก่อนอื่นให้ใช้ $H$...

$$
\begin{equation*}
    \longrightarrow\quad\frac{1}{2}|0\rangle\Big( |\psi_j\rangle + P|\psi_i\rangle\Big) + \frac{1}{2}|1\rangle\Big(|\psi_j\rangle - P|\psi_i\rangle\Big)
\end{equation*}
$$

...จากนั้นวัด:

$$
\begin{equation*}
\begin{split}
    \Rightarrow\quad\langle X\rangle &= \frac{1}{4}\Bigg(\Big\|| \psi_j\rangle + P|\psi_i\rangle \Big\|^2-\Big\||\psi_j\rangle - P|\psi_i\rangle\Big\|^2\Bigg) \\
    &= \text{Re}\Big[\langle\psi_j| P|\psi_i\rangle\Big].
\end{split}
\end{equation*}
$$

จากเอกลักษณ์ $|a + b\|^2 = \langle a + b | a + b \rangle = \|a\|^2 + \|b\|^2 + 2\text{Re}\langle a | b \rangle$ ในทำนองเดียวกัน การวัด $Y$ ให้ผลลัพธ์

$$
\begin{equation*}
    \langle Y\rangle = \text{Im}\Big[\langle\psi_j| P|\psi_i\rangle\Big].
\end{equation*}
$$

In [11]:
ordering_comparison_rows = []
infidelity_reps = [1, 2, 4, 8]

for ordering, ham_ordered in hamiltonians_by_ordering.items():
    for reps in infidelity_reps:
        circuit = build_numeric_evolution_circuit(
            ham_ordered,
            LieTrotter,
            comparison_time,
            reps=reps,
        )
        decomposed_circuit = simple_transpilation(circuit)
        infidelity = process_infidelity(decomposed_circuit, exact_matrix)
        ordering_comparison_rows.append(
            {
                "ordering": ordering,
                "synthesis": f"LieTrotter(reps={reps})",
                "infidelity": infidelity,
                **summarize_circuit(decomposed_circuit),
            }
        )
        if reps == 1:
            print(f"Circuit for {ordering} ordering:")
            print([op for op, _ in ham_ordered.to_list()])
            display(decomposed_circuit.draw("mpl", fold=-1, scale=0.6))

ordering_comparison_df = pd.DataFrame(ordering_comparison_rows)
display(ordering_comparison_df)

hamiltonian_for_synthesis = hamiltonians_by_ordering["even-odd edge-grouped"]

Circuit for naive ordering:
['IIIIIIIIIIXX', 'IIIIIIIIIXXI', 'IIIIIIIIXXII', 'IIIIIIIXXIII', 'IIIIIIXXIIII', 'IIIIIXXIIIII', 'IIIIXXIIIIII', 'IIIXXIIIIIII', 'IIXXIIIIIIII', 'IXXIIIIIIIII', 'XXIIIIIIIIII', 'IIIIIIIIIIYY', 'IIIIIIIIIYYI', 'IIIIIIIIYYII', 'IIIIIIIYYIII', 'IIIIIIYYIIII', 'IIIIIYYIIIII', 'IIIIYYIIIIII', 'IIIYYIIIIIII', 'IIYYIIIIIIII', 'IYYIIIIIIIII', 'YYIIIIIIIIII', 'IIIIIIIIIIZZ', 'IIIIIIIIIZZI', 'IIIIIIIIZZII', 'IIIIIIIZZIII', 'IIIIIIZZIIII', 'IIIIIZZIIIII', 'IIIIZZIIIIII', 'IIIZZIIIIIII', 'IIZZIIIIIIII', 'IZZIIIIIIIII', 'ZZIIIIIIIIII']


<Image src="../docs/images/tutorials/krylov-quantum-diagonalization/extracted-outputs/21eaab38-1.avif" alt="Output of the previous code cell" />

Circuit for even-then-odd ordering:
['IIIIIIIIIIXX', 'IIIIIIIIXXII', 'IIIIIIXXIIII', 'IIIIXXIIIIII', 'IIXXIIIIIIII', 'XXIIIIIIIIII', 'IIIIIIIIIXXI', 'IIIIIIIXXIII', 'IIIIIXXIIIII', 'IIIXXIIIIIII', 'IXXIIIIIIIII', 'IIIIIIIIIIYY', 'IIIIIIIIYYII', 'IIIIIIYYIIII', 'IIIIYYIIIIII', 'IIYYIIIIIIII', 'YYIIIIIIIIII', 'IIIIIIIIIYYI', 'IIIIIIIYYIII', 'IIIIIYYIIIII', 'IIIYYIIIIIII', 'IYYIIIIIIIII', 'IIIIIIIIIIZZ', 'IIIIIIIIZZII', 'IIIIIIZZIIII', 'IIIIZZIIIIII', 'IIZZIIIIIIII', 'ZZIIIIIIIIII', 'IIIIIIIIIZZI', 'IIIIIIIZZIII', 'IIIIIZZIIIII', 'IIIZZIIIIIII', 'IZZIIIIIIIII']


<Image src="../docs/images/tutorials/krylov-quantum-diagonalization/extracted-outputs/21eaab38-3.avif" alt="Output of the previous code cell" />

Circuit for even-odd edge-grouped ordering:
['IIIIIIIIIIXX', 'IIIIIIIIIIYY', 'IIIIIIIIIIZZ', 'IIIIIIIIXXII', 'IIIIIIIIYYII', 'IIIIIIIIZZII', 'IIIIIIXXIIII', 'IIIIIIYYIIII', 'IIIIIIZZIIII', 'IIIIXXIIIIII', 'IIIIYYIIIIII', 'IIIIZZIIIIII', 'IIXXIIIIIIII', 'IIYYIIIIIIII', 'IIZZIIIIIIII', 'XXIIIIIIIIII', 'YYIIIIIIIIII', 'ZZIIIIIIIIII', 'IIIIIIIIIXXI', 'IIIIIIIIIYYI', 'IIIIIIIIIZZI', 'IIIIIIIXXIII', 'IIIIIIIYYIII', 'IIIIIIIZZIII', 'IIIIIXXIIIII', 'IIIIIYYIIIII', 'IIIIIZZIIIII', 'IIIXXIIIIIII', 'IIIYYIIIIIII', 'IIIZZIIIIIII', 'IXXIIIIIIIII', 'IYYIIIIIIIII', 'IZZIIIIIIIII']


<Image src="../docs/images/tutorials/krylov-quantum-diagonalization/extracted-outputs/21eaab38-5.avif" alt="Output of the previous code cell" />

,ordering,synthesis,infidelity,depth,size,2q gates,2q depth
0,naive,LieTrotter(reps=1),0.999917,15,33,33,15
1,naive,LieTrotter(reps=2),0.805998,21,66,66,21
2,naive,LieTrotter(reps=4),0.271389,33,132,132,33
3,naive,LieTrotter(reps=8),0.074590,57,264,264,57
4,even-then-odd,LieTrotter(reps=1),0.999917,6,33,33,6
5,even-then-odd,LieTrotter(reps=2),0.805998,12,66,66,12
6,even-then-odd,LieTrotter(reps=4),0.271389,24,132,132,24
7,even-then-odd,LieTrotter(reps=8),0.074590,48,264,264,48
8,even-odd edge-grouped,LieTrotter(reps=1),0.998432,6,33,33,6
9,even-odd edge-grouped,LieTrotter(reps=2),0.653843,12,66,66,12


At `reps=1`, we observe that even-then-odd and even-odd edge-grouped orderings both collapse the depth from 15 to 6 by exposing parallel two-qubit layers, while the two-qubit gate count stays the same across all three orderings.
However, all three orderings have infidelity close to 1, so we sweep the number of Trotter repetitions to separate the orderings more clearly.
As the number of repetitions grows, the infidelity of `even-odd edge-grouped` ordering drops faster than the other two, reaching 0.045 at `reps=8` versus 0.075 for naive and even-then-odd ordering.

#### Compare product-formula synthesis

Next, we explore different advanced settings of Trotterization, with the fixed Hamiltonian ordering to the even-odd edge-grouped ordering.
We consider first-order Lie-Trotter, second-order Suzuki-Trotter, and fourth-order Suzuki-Trotter.

In [12]:
synthesis_comparison_rows = []

for num_trotter_steps in [1, 2, 3, 4, 5]:
    synthesis_cases = [
        ("LieTrotter", LieTrotter, {"reps": num_trotter_steps}),
        (
            "SuzukiTrotter(order=2)",
            SuzukiTrotter,
            {"order": 2, "reps": num_trotter_steps},
        ),
        (
            "SuzukiTrotter(order=4)",
            SuzukiTrotter,
            {"order": 4, "reps": num_trotter_steps},
        ),
    ]
    for label, synthesis, kwargs in synthesis_cases:
        circuit = build_numeric_evolution_circuit(
            hamiltonian_for_synthesis,
            synthesis,
            comparison_time,
            **kwargs,
        )
        decomposed_circuit = simple_transpilation(circuit)
        synthesis_comparison_rows.append(
            {
                "synthesis": label,
                "reps": kwargs["reps"],
                "infidelity": process_infidelity(
                    decomposed_circuit, exact_matrix
                ),
                **summarize_circuit(decomposed_circuit),
            }
        )

synthesis_comparison_df = pd.DataFrame(synthesis_comparison_rows)
display(synthesis_comparison_df.sort_values(["2q gates", "2q depth"]))

# For memory free
exact_matrix = None

,synthesis,reps,infidelity,depth,size,2q gates,2q depth
0,LieTrotter,1,9.984324e-01,6,33,33,6
1,SuzukiTrotter(order=2),1,9.733399e-01,9,51,51,9
3,LieTrotter,2,6.538427e-01,12,66,66,12
4,SuzukiTrotter(order=2),2,2.522533e-01,15,84,84,15
6,LieTrotter,3,3.197242e-01,18,99,99,18
7,SuzukiTrotter(order=2),3,4.804050e-02,21,117,117,21
9,LieTrotter,4,1.815291e-01,24,132,132,24
10,SuzukiTrotter(order=2),4,1.453103e-02,27,150,150,27
12,LieTrotter,5,1.161770e-01,30,165,165,30
2,SuzukiTrotter(order=4),1,2.884402e-01,33,183,183,33


![Output of the previous code cell](../docs/images/tutorials/krylov-quantum-diagonalization/extracted-outputs/7c1efca7-7db9-43a9-bcb7-053ba274d6f6-1.avif)

Circuit ของ Hadamard test อาจเป็น Circuit ที่ลึกมากเมื่อ decompose ไปเป็น native gates (ซึ่งจะเพิ่มขึ้นอีกหากคำนึงถึง topology ของอุปกรณ์)

In [13]:
controlled_extended_swap_test = extended_swap_test

# Reuse the ordering and synthesis rule selected by the comparison above.
evol_gate_optimized = PauliEvolutionGate(
    hamiltonian_for_synthesis,
    time=t,
    synthesis=SuzukiTrotter(order=2, reps=num_trotter_steps),
)

uncontrolled_evolution = QuantumCircuit(num_qubits, name="U_ST2(t)")
uncontrolled_evolution.append(evol_gate_optimized, range(num_qubits))

controlled_state_prep = QuantumCircuit(num_qubits + 1, name="C-Prep")
for q, bit in enumerate(reversed(ref_bitstring)):
    if bit == "1":
        controlled_state_prep.cx(ancilla, system_qubits[q])

vacuum_bitstring = "0" * num_qubits
vacuum_energy = basis_state_expectation_sparse(hamiltonian, "0" * num_qubits)

optimized_extended_swap_test = QuantumCircuit(num_qubits + 1)
optimized_extended_swap_test.h(ancilla)

# Prepare |psi_ref> only on the |1> branch.
optimized_extended_swap_test.compose(controlled_state_prep, inplace=True)
optimized_extended_swap_test.barrier()

# Apply the Trotterized time evolution without control.
optimized_extended_swap_test.compose(
    uncontrolled_evolution,
    qubits=system_qubits,
    inplace=True,
)
optimized_extended_swap_test.barrier()

# Map |0>|0...0> to |0>|psi_ref>, leaving the |1> branch unchanged.
optimized_extended_swap_test.x(ancilla)
optimized_extended_swap_test.compose(controlled_state_prep, inplace=True)
optimized_extended_swap_test.x(ancilla)

# Cancel the known vacuum phase so that the same X/Y observables can be used.
optimized_extended_swap_test.p(-vacuum_energy * t, ancilla)
optimized_extended_swap_test = simple_transpilation(
    optimized_extended_swap_test
)

print(f"Vacuum energy E_vac = {vacuum_energy:.1f}")
display(
    optimized_extended_swap_test.assign_parameters({t: 1.0}).draw(
        "mpl", scale=0.5, fold=26
    )
)

Vacuum energy E_vac = 11.0+0.0j


<Image src="../docs/images/tutorials/krylov-quantum-diagonalization/extracted-outputs/790e5e0c-1.avif" alt="Output of the previous code cell" />

#### Transpilation

Now, we transpile the controlled- and control-free circuits to become executable on the hardware.
Let's compare the result of the transpiled circuits.

In [14]:
pass_manager = generate_preset_pass_manager(
    backend=backend,
    optimization_level=3,
)

isa_controlled_extended_swap_test = pass_manager.run(
    controlled_extended_swap_test
)

pass_manager = generate_preset_pass_manager(
    backend=backend, optimization_level=3, routing_method="none"
)
isa_optimized_extended_swap_test = pass_manager.run(
    optimized_extended_swap_test
)

transpilation_result = [
    {
        "label": "abstract controlled U(t)",
        **summarize_circuit(isa_controlled_extended_swap_test),
    },
    {
        "label": "optimized non-controlled U(t)",
        **summarize_circuit(isa_optimized_extended_swap_test),
    },
]

display(pd.DataFrame(transpilation_result))


def filter_qubits_from_layout(layout):
    q_layout = Layout(
        {
            physical: virtual
            for physical, virtual in layout.get_physical_bits().items()
            if virtual._register.name == "q"
        }
    )
    return q_layout


print(
    filter_qubits_from_layout(
        isa_optimized_extended_swap_test.layout.initial_layout
    )
)

isa_observables = [
    op.apply_layout(isa_optimized_extended_swap_test.layout)
    for op in observables
]

,label,depth,size,2q gates,2q depth
0,abstract controlled U(t),15457,23786,4686,4580
1,optimized non-controlled U(t),261,1716,307,57


Layout({
18: <Qubit register=(13, "q"), index=0>,
5: <Qubit register=(13, "q"), index=1>,
6: <Qubit register=(13, "q"), index=2>,
7: <Qubit register=(13, "q"), index=3>,
8: <Qubit register=(13, "q"), index=4>,
9: <Qubit register=(13, "q"), index=5>,
10: <Qubit register=(13, "q"), index=6>,
11: <Qubit register=(13, "q"), index=7>,
12: <Qubit register=(13, "q"), index=8>,
13: <Qubit register=(13, "q"), index=9>,
14: <Qubit register=(13, "q"), index=10>,
15: <Qubit register=(13, "q"), index=11>,
19: <Qubit register=(13, "q"), index=12>
})


## ขั้นตอนที่ 2: ปรับปรุงปัญหาให้เหมาะสมสำหรับการรันบน Quantum hardware
### Hadamard test แบบมีประสิทธิภาพ
เราสามารถปรับปรุง Circuit ที่ลึกมากสำหรับ Hadamard test ที่ได้มาได้ โดยการนำการประมาณค่าบางอย่างมาใช้และอาศัยสมมติฐานบางประการเกี่ยวกับ Hamiltonian ของโมเดล ตัวอย่างเช่น ลองพิจารณา Circuit ต่อไปนี้สำหรับ Hadamard test:

![fig3.png](../docs/images/tutorials/krylov-subspace-diagonalization/35b13797-5a46-486c-b50e-97c205cc9747.avif)

สมมติว่าเราสามารถคำนวณ $E_0$ ซึ่งเป็น eigenvalue ของ $|0\rangle^N$ ภายใต้ Hamiltonian $H$ ได้แบบคลาสสิก
เงื่อนไขนี้จะเป็นจริงเมื่อ Hamiltonian รักษา U(1) symmetry แม้ว่านี่อาจดูเหมือนเป็นสมมติฐานที่เข้มแข็ง แต่มีหลายกรณีที่ปลอดภัยที่จะสมมติว่ามี vacuum state (ในกรณีนี้จะ map ไปยัง state $|0\rangle^N$) ซึ่งไม่ได้รับผลกระทบจาก Hamiltonian ตัวอย่างเช่น Hamiltonian ทางเคมีที่อธิบายโมเลกุลที่เสถียร (ซึ่งจำนวนอิเล็กตรอนได้รับการอนุรักษ์) เป็นไปตามเงื่อนไขนี้
เมื่อกำหนดว่า Gate $\text{Prep} \; \psi$ เตรียม reference state ที่ต้องการ $\ket{psi} = \text{Prep} \; \psi \ket{0} = e^{-i H 0 dt} U_{\psi} \ket{0}$ เช่น เพื่อเตรียม HF state สำหรับเคมี $\text{Prep} \; \psi$ จะเป็นผลคูณของ single-qubit NOT ดังนั้น controlled-$\text{Prep} \; \psi$ จะเป็นแค่ผลคูณของ CNOT
จากนั้น Circuit ด้านบนจะ implement state ต่อไปนี้ก่อนการวัด:

$$
\begin{equation}
\begin{split}
    \ket{0} \ket{0}^N\xrightarrow{H}&\frac{1}{\sqrt{2}}
    \left(
    \ket{0}\ket{0}^N+ \ket{1} \ket{0}^N
    \right)\\
    \xrightarrow{\text{1-ctrl-init}}&\frac{1}{\sqrt{2}}\left(|0\rangle|0\rangle^N+|1\rangle|\psi\rangle\right)\\
    \xrightarrow{U}&\frac{1}{\sqrt{2}}\left(e^{i\phi}\ket{0}\ket{0}^N+\ket{1} U\ket{\psi}\right)\\
    \xrightarrow{\text{0-ctrl-init}}&\frac{1}{\sqrt{2}}
    \left(
    e^{i\phi}\ket{0} \ket{\psi}
    +\ket{1} U\ket{\psi}
    \right)\\
    =&\frac{1}{2}
    \left(
    \ket{+}\left(e^{i\phi}\ket{\psi}+U\ket{\psi}\right)
    +\ket{-}\left(e^{i\phi}\ket{\psi}-U\ket{\psi}\right)
    \right)\\
    =&\frac{1}{2}
    \left(
    \ket{+i}\left(e^{i\phi}\ket{\psi}-iU\ket{\psi}\right)
    +\ket{-i}\left(e^{i\phi}\ket{\psi}+iU\ket{\psi}\right)
    \right)
\end{split}
\end{equation}
$$

โดยเราได้ใช้ phase shift ที่จำลองแบบคลาสสิกได้ $ U\ket{0}^N = e^{i\phi}\ket{0}^N$ ในบรรทัดที่สาม ดังนั้น expectation value จึงได้รับเป็น

$$
\begin{equation}
\begin{split}
    \langle X\otimes P\rangle&=\frac{1}{4}
    \Big(
    \left(e^{-i\phi}\bra{\psi}+\bra{\psi}U^\dagger\right)P\left(e^{i\phi}\ket{\psi}+U\ket{\psi}\right)
    \\
    &\qquad-\left(e^{-i\phi}\bra{\psi}-\bra{\psi}U^\dagger\right)P\left(e^{i\phi}\ket{\psi}-U\ket{\psi}\right)
    \Big)\\
    &=\text{Re}\left[e^{-i\phi}\bra{\psi}PU\ket{\psi}\right],
\end{split}
\end{equation}
$$

$$
\begin{equation}
\begin{split}
    \langle Y\otimes P\rangle&=\frac{1}{4}
    \Big(
    \left(e^{-i\phi}\bra{\psi}+i\bra{\psi}U^\dagger\right)P\left(e^{i\phi}\ket{\psi}-iU\ket{\psi}\right)
    \\
    &\qquad-\left(e^{-i\phi}\bra{\psi}-i\bra{\psi}U^\dagger\right)P\left(e^{i\phi}\ket{\psi}+iU\ket{\psi}\right)
    \Big)\\
    &=\text{Im}\left[e^{-i\phi}\bra{\psi}PU\ket{\psi}\right].
\end{split}
\end{equation}
$$

ด้วยสมมติฐานเหล่านี้ เราสามารถเขียน expectation value ของ operator ที่สนใจโดยใช้ controlled operations น้อยลง ที่จริงแล้ว เราต้องการ implement เฉพาะ controlled state preparation $\text{Prep} \; \psi$ เท่านั้น ไม่ใช่ controlled time evolution การกำหนดกรอบการคำนวณใหม่ดังกล่าวจะช่วยให้เราลด depth ของ Circuit ที่ได้ออกมาได้อย่างมาก

### แยกสลาย time-evolution operator ด้วย Trotter decomposition
แทนที่จะ implement time-evolution operator แบบแม่นยำ เราสามารถใช้ Trotter decomposition เพื่อ implement การประมาณค่าของมันได้ การทำซ้ำ Trotter decomposition ลำดับหนึ่งหลายครั้งจะช่วยลด error ที่เกิดจากการประมาณค่าได้อีกด้วย ต่อไปนี้ เราจะสร้าง Trotter implementation โดยตรงในวิธีที่มีประสิทธิภาพสูงสุดสำหรับ interaction graph ของ Hamiltonian ที่เรากำลังพิจารณา (มีเฉพาะ nearest neighbor interactions) ในทางปฏิบัติ เราแทรก Pauli rotations $R_{xx}$, $R_{yy}$, $R_{zz}$ ด้วย parametrized angle $t$ ซึ่งสอดคล้องกับการ implement โดยประมาณของ $e^{-i (XX + YY + ZZ) t}$ เนื่องจากความแตกต่างในนิยามของ Pauli rotations และ time-evolution ที่เราพยายาม implement เราจึงต้องใช้พารามิเตอร์ $2*dt$ เพื่อให้ได้ time-evolution ของ $dt$ นอกจากนี้ เราสลับลำดับของ operations สำหรับ Trotter step ที่เป็นเลขคี่ ซึ่งให้ผลลัพธ์เทียบเท่าแต่ช่วยให้สังเคราะห์ operations ที่อยู่ติดกันเป็น unitary เดียวของ $SU(2)$ ได้ ส่งผลให้ได้ Circuit ที่ตื้นกว่ามากเมื่อเทียบกับการใช้ฟังก์ชัน `PauliEvolutionGate()` แบบทั่วไป

In [15]:
pub_list = []
d_values = list(range(1, krylov_dim))

# Exact local statevector estimator.
estimator = StatevectorEstimator()

# We use the circuit before the transpilation for the local simulator,
# but we will use the transpiled circuit for the real backend.
for d in d_values:
    parameter_values = [d * dt]

    for ob in observables:
        pub_list.append(
            (
                optimized_extended_swap_test,
                ob,
                parameter_values,
            )
        )

job = estimator.run(pub_list)

# Local PrimitiveJob does not provide Runtime-style job inputs,
# so preserve the inputs directly.
inputs = pub_list
result = job.result()

print(f"Number of Krylov basis states: r = {len(d_values)}")
print(f"Number of PUBs: {len(pub_list)}")
print(f"Each d uses observables: {observable_labels}")

Number of Krylov basis states: r = 9
Number of PUBs: 36
Each d uses observables: ['Re S_0d', 'Im S_0d', 'Re shifted H_0d', 'Im shifted H_0d']


![Output of the previous code cell](../docs/images/tutorials/krylov-quantum-diagonalization/extracted-outputs/267716dc-fa23-41bd-abe4-6d4e0499a0f4-0.avif)

### ใช้ Circuit ที่ปรับปรุงแล้วสำหรับการเตรียม state

In [16]:
h_shifted_row_est = np.zeros(krylov_dim, dtype=complex)
s_row_est = np.zeros(krylov_dim, dtype=complex)

h_shifted_row_est[0] = ref_energy - shift_tau
s_row_est[0] = 1.0

for idx, (pub_input, pub_result) in enumerate(zip(inputs, result)):
    d_index, obs_index = divmod(idx, len(observables))
    ev = np.asarray(pub_result.data.evs).reshape(-1)[0]
    std = np.asarray(pub_result.data.stds).reshape(-1)[0]

    if obs_index == 0:
        s_row_est[d_index + 1] = ev
    elif obs_index == 1:
        s_row_est[d_index + 1] += 1j * ev
    elif obs_index == 2:
        h_shifted_row_est[d_index + 1] = ev
    elif obs_index == 3:
        h_shifted_row_est[d_index + 1] += 1j * ev

# H_0d = shifted_H_0d + tau * S_0d.
h_row_est = h_shifted_row_est + shift_tau * s_row_est

h_matrix_est = la.toeplitz(h_row_est.conj(), h_row_est)
s_matrix_est = la.toeplitz(s_row_est.conj(), s_row_est)

s_eigvals = la.eigvalsh(0.5 * (s_matrix_est + s_matrix_est.conj().T))
positive_s_eigvals = s_eigvals[s_eigvals > 1e-12]
s_condition_number = (
    positive_s_eigvals[-1] / positive_s_eigvals[0]
    if len(positive_s_eigvals) > 0
    else np.inf
)


with np.printoptions(precision=3, suppress=True):
    print("Estimated first row of S:")
    print(s_row_est)
    print()
    print("Estimated first row of H:")
    print(h_row_est)
    print()
    print("Eigenvalues of the estimated overlap matrix S:")
    print(s_eigvals)
    print(f"Condition number above 1e-12: {s_condition_number:.3e}")
    print()

Estimated first row of S:
[ 1.   +0.j     0.758-0.596j  0.203-0.836j -0.291-0.636j -0.444-0.229j
 -0.276+0.053j -0.044+0.051j  0.006-0.121j -0.155-0.218j -0.345-0.101j]

Estimated first row of H:
[ 7.   +0.j     4.842-4.76j   0.044-6.185j -3.791-3.653j -4.137+0.39j
 -1.495+2.654j  1.331+1.777j  1.85 -0.763j -0.032-2.278j -2.222-1.372j]

Eigenvalues of the estimated overlap matrix S:
[-0.     0.     0.     0.     0.     0.     0.01   0.355  3.526  6.109]
Condition number above 1e-12: 1.904e+12



![Output of the previous code cell](../docs/images/tutorials/krylov-quantum-diagonalization/extracted-outputs/70411715-eed3-4cf5-961d-06a6f1e04efc-0.avif)

### Template circuits สำหรับคำนวณ matrix elements ของ $\tilde{S}$ และ $\tilde{H}$ ผ่าน Hadamard test
ความแตกต่างเพียงอย่างเดียวระหว่าง Circuit ที่ใช้ใน Hadamard test คือ phase ใน time-evolution operator และ observable ที่วัด ดังนั้นเราจึงสามารถเตรียม template circuit ซึ่งแทน Circuit ทั่วไปสำหรับ Hadamard test โดยมี placeholder สำหรับ Gate ที่ขึ้นอยู่กับ time-evolution operator

In [17]:
exact_evals = diagonalize_single_1_subspace(hamiltonian)
exact_ground = min(exact_evals)
print("exact ground state energy:  ", exact_ground)

threshold = 1e-12
energy_convergence = []

for r in range(1, krylov_dim + 1):
    energy_est_kqd, coeffs_est, retained_est = solve_thresholded_gevp(
        h_matrix_est[:r, :r],
        s_matrix_est[:r, :r],
        threshold=threshold,
    )
    energy_convergence.append(energy_est_kqd)
    print(
        f"Krylov ground state energy (dim={r}, retained={retained_est}): ",
        energy_est_kqd,
    )

exact ground state energy:   3.136296694843727
Krylov ground state energy (dim=1, retained=1):  7.0
Krylov ground state energy (dim=2, retained=2):  4.184510657551266
Krylov ground state energy (dim=3, retained=3):  3.5539074630394136
Krylov ground state energy (dim=4, retained=4):  3.3366270761341044
Krylov ground state energy (dim=5, retained=5):  3.252017453225087
Krylov ground state energy (dim=6, retained=6):  3.2300275138879186
Krylov ground state energy (dim=7, retained=7):  3.2299154099085685
Krylov ground state energy (dim=8, retained=7):  3.2298063744216776
Krylov ground state energy (dim=9, retained=7):  3.2296778282872456
Krylov ground state energy (dim=10, retained=8):  3.223647515867734


In [18]:
def plot_energy_convergence(energy_convergence, exact_ground, krylov_dim):
    fig, ax = plt.subplots(figsize=(7, 4.5))

    ax.plot(
        range(1, krylov_dim + 1),
        energy_convergence,
        marker="o",
        label="KQD estimate",
    )

    ax.axhline(
        exact_ground,
        linestyle="--",
        label=f"Exact ground energy = {exact_ground:.6f}",
    )

    ax.set_xlabel("Krylov dimension")
    ax.set_ylabel("Ground-state energy")
    ax.grid(True, alpha=0.3)
    ax.legend()
    plt.tight_layout()
    plt.show()


plot_energy_convergence(energy_convergence, exact_ground, krylov_dim)

<Image src="../docs/images/tutorials/krylov-quantum-diagonalization/extracted-outputs/08d57f33-0.avif" alt="Output of the previous code cell" />

## Large-scale hardware example

The previous section used a 12-qubit model so that the statevector simulation could be used as a diagnostic. We now scale the same KQD workflow to a 30-qubit Heisenberg chain and prepare the workload for execution on IBM Quantum hardware.

### Steps 1-4 compressed into a single code block

Here we now put all of these details together into a singular workflow at a larger scale, which is then run on our real quantum hardware. In this section, we apply realistic error-mitigation settings to improve the reliability of the results. Since the matrix elements corresponding to different values of $d$ can be evaluated in parallel, we use `Batch` mode to execute them efficiently.

In [ ]:
# -------------------------Step 1-------------------------

# Map the classical problem to quantum circuits and observables.
# Problem and KQD parameters.
large_num_qubits = 30
large_krylov_dim = 7
large_num_trotter_steps = 3
large_dt = np.pi / (3 * (large_num_qubits - 1))
large_t = Parameter("t_large")

# Use a single excitation near the center of the chain.
large_excitation_qubit = large_num_qubits // 2
large_ref_label = ["0"] * large_num_qubits
large_ref_label[large_num_qubits - 1 - large_excitation_qubit] = "1"
large_ref_bitstring = "".join(large_ref_label)

# Use the ordering and product formula selected in the preceding section.
large_hamiltonian = make_heisenberg_hamiltonian_ordered(
    large_num_qubits,
    ordering="even-odd edge-grouped",
    coupling=1.0,
)
large_ref_energy = float(
    np.real(
        basis_state_expectation_sparse(large_hamiltonian, large_ref_bitstring)
    )
)
large_vacuum_energy = float(
    np.real(
        basis_state_expectation_sparse(
            large_hamiltonian, "0" * large_num_qubits
        )
    )
)

large_evolution_gate = PauliEvolutionGate(
    large_hamiltonian,
    time=large_t,
    synthesis=SuzukiTrotter(order=2, reps=large_num_trotter_steps),
)
large_uncontrolled_evolution = QuantumCircuit(
    large_num_qubits,
    name="U_ST2_large(t)",
)
large_uncontrolled_evolution.append(
    large_evolution_gate,
    range(large_num_qubits),
)

# Build the control-free extended-swap-test circuit.
large_ancilla = 0
large_system_qubits = list(range(1, large_num_qubits + 1))

large_controlled_state_prep = QuantumCircuit(
    large_num_qubits + 1,
    name="C-Prep-large",
)
large_controlled_state_prep.cx(
    large_ancilla,
    large_system_qubits[large_excitation_qubit],
)

large_extended_swap_test = QuantumCircuit(large_num_qubits + 1)
large_extended_swap_test.h(large_ancilla)
large_extended_swap_test.compose(large_controlled_state_prep, inplace=True)
large_extended_swap_test.compose(
    large_uncontrolled_evolution,
    qubits=large_system_qubits,
    inplace=True,
)
large_extended_swap_test.x(large_ancilla)
large_extended_swap_test.compose(
    large_controlled_state_prep.inverse(),
    inplace=True,
)
large_extended_swap_test.x(large_ancilla)
large_extended_swap_test.p(
    -large_vacuum_energy * large_t,
    large_ancilla,
)

# Reuse the Hamiltonian-shifting construction from the preceding section.
(
    large_obs_x_shifted_hamiltonian,
    large_obs_y_shifted_hamiltonian,
    large_shift_tau,
) = make_reduced_heisenberg_observables(large_ref_bitstring)

large_observables = [
    SparsePauliOp("I" * large_num_qubits + "X"),
    SparsePauliOp("I" * large_num_qubits + "Y"),
    large_obs_x_shifted_hamiltonian,
    large_obs_y_shifted_hamiltonian,
]
large_observable_labels = [
    "Re S_0d",
    "Im S_0d",
    "Re shifted H_0d",
    "Im shifted H_0d",
]

# -------------------------Step 2-------------------------

# Optimize the problem for quantum execution.
# Select a real backend and transpile the parameterized circuit to ISA form.
large_backend = service.backend("ibm_boston")
large_pass_manager = generate_preset_pass_manager(
    backend=large_backend, optimization_level=3, routing_method="none"
)
large_isa_circuit = large_pass_manager.run(large_extended_swap_test)
large_isa_observables = [
    observable.apply_layout(large_isa_circuit.layout)
    for observable in large_observables
]

large_two_qubit_gate_count = sum(
    instruction.operation.num_qubits == 2
    for instruction in large_isa_circuit.data
)

print(f"Backend: {large_backend.name}")
print(f"System qubits: {large_num_qubits}")
print(f"Total circuit qubits: {large_isa_circuit.num_qubits}")
print(f"Krylov dimension: {large_krylov_dim}")
print(f"Time step: {large_dt:.6f}")
print(f"Shift tau: {large_shift_tau}")
print(f"ISA circuit depth: {large_isa_circuit.depth()}")
print(f"ISA two-qubit gates: {large_two_qubit_gate_count}")
print(
    f"ISA two-qubit depth: {large_isa_circuit.depth(lambda x: x[0].num_qubits == 2)}"
)

# -------------------------Step 3-------------------------

# Execute on quantum hardware with Qiskit Runtime primitives.
# Submit one job per d, with all four observables in that job.
large_d_values = list(range(1, large_krylov_dim))
retrieve_batch_id = None
large_jobs = []

if retrieve_batch_id is None:
    large_estimator_options = {
        "default_shots": 8192,
        "dynamical_decoupling": {
            "enable": True,
            "sequence_type": "XpXm",
        },
        "resilience": {
            "measure_mitigation": True,
            "measure_noise_learning": {
                "num_randomizations": 32,
                "shots_per_randomization": 256,
            },
            "layer_noise_learning": {
                "max_layers_to_learn": 4,
                "layer_pair_depths": [0, 1, 2, 4, 16, 32],
                "num_randomizations": 32,
                "shots_per_randomization": 128,
            },
            "zne_mitigation": True,
            "zne": {
                "amplifier": "pea",
                "noise_factors": [1.0, 1.5, 2.0],
                "extrapolator": ("exponential", "linear"),
            },
        },
        "twirling": {
            "enable_gates": True,
            "enable_measure": True,
            "num_randomizations": 32,
            "shots_per_randomization": 256,
            "strategy": "active-accum",
        },
    }

    with Batch(backend=large_backend) as large_batch:
        large_batch_id = large_batch.session_id
        large_estimator = EstimatorV2(
            mode=large_batch,
            options=large_estimator_options,
        )
        # Krylov quantum diagonalization of lattice Hamiltonians -> TUT_KQDOLH.
        large_estimator.options.environment.job_tags = ["TUT_KQDOLH"]

        for d in large_d_values:
            parameter_values = [d * large_dt]
            pubs_for_d = [
                (large_isa_circuit, observable, parameter_values)
                for observable in large_isa_observables
            ]
            job = large_estimator.run(pubs_for_d)
            large_jobs.append(job)
            print(
                f"Submitted d={d}: job_id={job.job_id()}, "
                f"PUBs={len(pubs_for_d)}"
            )

    print(f"Batch ID: {large_batch_id}")
else:
    large_batch_id = retrieve_batch_id
    large_jobs = service.jobs(
        session_id=large_batch_id,
        limit=None,
        descending=False,
    )

large_job_ids_by_d = {
    d: job.job_id() for d, job in zip(large_d_values, large_jobs)
}
print(f"Job IDs by d: {large_job_ids_by_d}")

# -------------------------Step 4-------------------------

# Post-process the quantum results and solve the classical GEVP.
# Reconstruct the first rows of S and the shifted Hamiltonian matrix.
large_s_row_est = np.zeros(large_krylov_dim, dtype=complex)
large_h_shifted_row_est = np.zeros(large_krylov_dim, dtype=complex)
large_s_row_est[0] = 1.0
large_h_shifted_row_est[0] = large_ref_energy - large_shift_tau

for d, job in zip(large_d_values, large_jobs):
    job_result = job.result()

    if len(job_result) != len(large_observables):
        raise RuntimeError(
            f"Expected {len(large_observables)} PUB results for d={d}, "
            f"but received {len(job_result)}."
        )

    expectation_values = [
        np.asarray(pub_result.data.evs).reshape(-1)[0]
        for pub_result in job_result
    ]

    large_s_row_est[d] = expectation_values[0] + 1j * expectation_values[1]
    large_h_shifted_row_est[d] = (
        expectation_values[2] + 1j * expectation_values[3]
    )

# H_0d = shifted_H_0d + tau * S_0d.
large_h_row_est = large_h_shifted_row_est + large_shift_tau * large_s_row_est

large_s_matrix_est = la.toeplitz(
    large_s_row_est.conj(),
    large_s_row_est,
)
large_h_matrix_est = la.toeplitz(
    large_h_row_est.conj(),
    large_h_row_est,
)

large_exact_gnd = min(diagonalize_single_1_subspace(large_hamiltonian))

large_energy_convergence = []

with np.printoptions(precision=5, suppress=True):
    print("Estimated first row of S:")
    print(large_s_row_est)
    print("Estimated first row of H:")
    print(large_h_row_est)
    print("exact ground state energy:  ", large_exact_gnd)

    for r in range(1, large_krylov_dim + 1):
        energy_est_kqd, coeffs_est, retained_est = solve_thresholded_gevp(
            large_h_matrix_est[:r, :r],
            large_s_matrix_est[:r, :r],
            threshold=5e-2,
        )
        large_energy_convergence.append(energy_est_kqd)
        print(
            f"Krylov ground state energy (dim={r}, retained={retained_est}): ",
            energy_est_kqd,
        )

plot_energy_convergence(
    large_energy_convergence, large_exact_gnd, large_krylov_dim
)

Backend: ibm_boston
System qubits: 30
Total circuit qubits: 156
Krylov dimension: 7
Time step: 0.036110
Shift tau: 25.0
ISA circuit depth: 219
ISA two-qubit gates: 728
ISA two-qubit depth: 52
Job IDs by d: {1: 'd9fl4p4jeosc73fk4dg0', 2: 'd9fl4pineu4c739poecg', 3: 'd9fl4q2neu4c739poedg', 4: 'd9fl4qhhtsac739fjgn0', 5: 'd9fl4r4jeosc73fk4dhg', 6: 'd9fl4rkjeosc73fk4dj0'}
Estimated first row of S:
[ 1.     +0.j       0.42055-0.65466j -0.17883-0.73125j -0.64523-0.31601j
 -0.70781+0.55574j -0.02467+0.70525j  0.59301+0.55602j]
Estimated first row of H:
[ 25.      +0.j       10.3419 -16.5586j   -5.017  -18.03933j
 -16.44797 -6.98173j -17.17081+14.84502j   0.62722+17.81196j
  15.68886+12.74875j]
exact ground state energy:   21.021912418526902
Krylov ground state energy (dim=1, retained=1):  25.0
Krylov ground state energy (dim=2, retained=2):  24.432409110686205
Krylov ground state energy (dim=3, retained=3):  24.256856893278556
Krylov ground state energy (dim=4, retained=4):  23.727409826799715


<Image src="../docs/images/tutorials/krylov-quantum-diagonalization/extracted-outputs/e3e0d1f3-1.avif" alt="Output of the previous code cell" />

## Appendix: The Hamiltonian-function (spectral-filter) viewpoint

The main workflow presented KQD operationally: build a Krylov basis from real-time-evolved states, estimate the projected matrices $\mathcal{H}$ and $\mathcal{S}$, and solve the GEVP.
This appendix revisits the *same* computation from a complementary angle that explains **why** KQD works: the *Hamiltonian-function*, or *spectral-filter*, viewpoint [\[3\]](#references), [\[5\]](#references).
It reuses the 12-qubit model, time step $\Delta t$, and the Krylov solution already obtained above; no new circuit execution is required.

### The reference state as an energy distribution

Let the Hamiltonian have the eigendecomposition

$$
\begin{equation*}
H=\sum_{m} E_m\,|E_m\rangle\!\langle E_m|,
\qquad E_0\le E_1\le\cdots,
\end{equation*}
$$

with energy eigenstates $|E_m\rangle$.
Any reference state can be expanded in this eigenbasis,

$$
\begin{equation*}
|\psi_{0}\rangle=\sum_m a_m\,|E_m\rangle,
\qquad a_m=\langle E_m|\psi_{0}\rangle,
\end{equation*}
$$

so it carries a **spectral weight** $p_m=|a_m|^2$ at each energy $E_m$.
The reference energy is the mean of this distribution, $\langle H\rangle_{0}=\sum_m p_m E_m$.

The eigendecomposition of a general $n$-qubit Hamiltonian is exponentially costly, so this picture is only a diagnostic, which is *not* part of the algorithm.
Here, however, we can compute it cheaply for the *same* $n=12$ problem: the Heisenberg Hamiltonian conserves the total excitation number, and the reference $|000001000000\rangle$ carries a single excitation, so its entire spectral content lives in the single-excitation subspace whose dimension grows only linearly with $n$.
We therefore reuse the exact single-excitation block (already used above as a benchmark) and read off the reference distribution $\{(E_m, p_m)\}$ inside that subspace.

In [20]:
# Exact single-excitation-subspace decomposition of the reference state.
# This reuses make_heisenberg_hamiltonian / _basis_state_transition_amplitude_sparse
# and the n=12 `hamiltonian` and `ref_bitstring` defined in the small-scale example.
single_excitation_states = [1 << k for k in range(num_qubits)]

h_single = np.array(
    [
        [
            _basis_state_transition_amplitude_sparse(hamiltonian, bra, ket)
            for ket in single_excitation_states
        ]
        for bra in single_excitation_states
    ]
)
h_single = 0.5 * (h_single + h_single.conj().T)

subspace_evals, subspace_evecs = la.eigh(h_single)
subspace_evals = np.real(subspace_evals)

# Reference-state coordinates inside the single-excitation subspace.
ref_position = single_excitation_states.index(int(ref_bitstring, 2))
ref_in_subspace = np.zeros(num_qubits, dtype=complex)
ref_in_subspace[ref_position] = 1.0

# Amplitudes and spectral weights of the reference in the energy eigenbasis.
ref_eigen_amplitudes = subspace_evecs.conj().T @ ref_in_subspace
ref_spectral_weights = np.abs(ref_eigen_amplitudes) ** 2

print(f"Single-excitation subspace dimension: {num_qubits}")
print(f"Subspace ground-state energy:         {subspace_evals[0]:.6f}")
print(
    f"Reference energy (sum p_m E_m):       {np.sum(ref_spectral_weights * subspace_evals):.6f}"
)
print(f"Reference weight on subspace ground:  {ref_spectral_weights[0]:.6f}")

Single-excitation subspace dimension: 12
Subspace ground-state energy:         3.136297
Reference energy (sum p_m E_m):       7.000000
Reference weight on subspace ground:  0.163827


เราลด depth ของ Hadamard test ได้อย่างมาก ด้วยการรวม Trotter approximation และ uncontrolled unitaries เข้าด้วยกัน

## ขั้นตอนที่ 3: รันด้วย Qiskit primitives
สร้าง instance ของ Backend และกำหนดพารามิเตอร์ runtime

In [21]:
def trigonometric_krylov_filter(
    coeffs: np.ndarray,
    energies: np.ndarray,
    time_step: float,
) -> np.ndarray:
    """Evaluate the learned Krylov filter f(E) = sum_l c_l exp(-i l dt E)."""
    values = np.zeros_like(energies, dtype=complex)
    for ell, coeff in enumerate(coeffs):
        values += coeff * np.exp(-1j * ell * time_step * energies)
    return values


def filtered_spectral_weights(
    weights: np.ndarray,
    filter_values: np.ndarray,
) -> np.ndarray:
    """Reshape spectral weights by |f(E)|^2 and renormalize."""
    reshaped = weights * np.abs(filter_values) ** 2
    return reshaped / np.sum(reshaped)


# Recover the KQD coefficients from the already-estimated projected matrices.
# The shift only moves H by tau * S, so it does not change the GEVP eigenvector;
# we solve at the full Krylov dimension used in the small-scale example.
_, kqd_coeffs, _ = solve_thresholded_gevp(
    h_matrix_est,
    s_matrix_est,
    threshold=1e-12,
)

filter_on_spectrum = trigonometric_krylov_filter(
    kqd_coeffs, subspace_evals, dt
)
filtered_weights = filtered_spectral_weights(
    ref_spectral_weights, filter_on_spectrum
)

print(f"Ground-state overlap  (reference): {ref_spectral_weights[0]:.4f}")
print(f"Ground-state overlap  (filtered):  {filtered_weights[0]:.4f}")
print(
    f"Mean energy           (reference): {np.sum(ref_spectral_weights * subspace_evals):.6f}"
)
print(
    f"Mean energy           (filtered):  {np.sum(filtered_weights * subspace_evals):.6f}"
)

Ground-state overlap  (reference): 0.1638
Ground-state overlap  (filtered):  0.9638
Mean energy           (reference): 7.000000
Mean energy           (filtered):  3.164503


### Transpile ไปยัง QPU
ก่อนอื่น เราเลือก subset ของ coupling map ที่มี Qubit "คุณภาพดี" (โดยที่ "ดี" ที่นี่ค่อนข้างยืดหยุ่น เราต้องการหลีกเลี่ยง Qubit ที่ทำงานได้แย่มากเป็นหลัก) และสร้าง target ใหม่สำหรับการ Transpile

In [22]:
fig, ax = plt.subplots(figsize=(8, 4))

visible = (ref_spectral_weights > 1e-4) | (filtered_weights > 1e-4)
ax.bar(
    subspace_evals[visible],
    ref_spectral_weights[visible],
    width=0.18,
    alpha=0.45,
    label="reference $p_m$",
)
ax.bar(
    subspace_evals[visible],
    filtered_weights[visible],
    width=0.14,
    alpha=0.9,
    label=r"filtered $p_m\,|f_{\rm KQD}(E_m)|^2$",
)

energy_grid = np.linspace(
    subspace_evals.min() - 0.5, subspace_evals.max() + 0.5, 800
)
filter_intensity = (
    np.abs(trigonometric_krylov_filter(kqd_coeffs, energy_grid, dt)) ** 2
)
filter_intensity /= filter_intensity.max()
ax.plot(
    energy_grid,
    filter_intensity,
    color="k",
    linewidth=2,
    label=r"$|f_{\rm KQD}(E)|^2$ (normalized)",
)

ax.axvline(
    subspace_evals[0],
    color="C3",
    linestyle="--",
    linewidth=1,
    label="subspace ground energy",
)
ax.set_xlabel("Energy eigenvalue $E_m$")
ax.set_ylabel("Spectral weight")
ax.set_ylim(0, 1)
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()

<Image src="../docs/images/tutorials/krylov-quantum-diagonalization/extracted-outputs/app-code-plot-0.avif" alt="Output of the previous code cell" />

จากนั้น Transpile virtual Circuit ไปยัง physical layout ที่ดีที่สุดใน target ใหม่นี้

In [23]:
# Sweep the Krylov dimension using the leading r x r blocks of the estimated matrices.
sweep_dims = [r for r in (2, 4, 6, 8, krylov_dim) if r <= krylov_dim]
sweep_dims = sorted(set(sweep_dims))

energy_grid = np.linspace(
    subspace_evals.min() - 0.5, subspace_evals.max() + 0.5, 800
)

sweep_cases = []
print(" r    retained    ground overlap    filtered energy")
print("--    --------    --------------    ---------------")
for r in sweep_dims:
    _, coeffs_r, retained_r = solve_thresholded_gevp(
        h_matrix_est[:r, :r],
        s_matrix_est[:r, :r],
        threshold=1e-12,
    )
    filter_on_spectrum_r = trigonometric_krylov_filter(
        coeffs_r, subspace_evals, dt
    )
    filtered_weights_r = filtered_spectral_weights(
        ref_spectral_weights, filter_on_spectrum_r
    )
    filtered_energy_r = float(np.sum(filtered_weights_r * subspace_evals))

    sweep_cases.append((r, coeffs_r, filtered_weights_r))
    print(
        f"{r:2d}    {retained_r:8d}    {filtered_weights_r[0]:14.4f}    {filtered_energy_r:15.6f}"
    )

print(f"\nSubspace ground-state energy (target): {subspace_evals[0]:.6f}")

# One panel per Krylov dimension: filtered spectrum (bars) + filter intensity (curve).
fig, axes = plt.subplots(
    len(sweep_cases),
    1,
    figsize=(8, 2.1 * len(sweep_cases)),
    sharex=True,
)
axes = np.atleast_1d(axes)

for idx, (ax, (r, coeffs_r, filtered_weights_r)) in enumerate(
    zip(axes, sweep_cases)
):
    visible = (ref_spectral_weights > 1e-4) | (filtered_weights_r > 1e-4)
    ax.bar(
        subspace_evals[visible],
        ref_spectral_weights[visible],
        width=0.18,
        alpha=0.35,
        color="C0",
        label="reference $p_m$" if idx == 0 else None,
    )
    ax.bar(
        subspace_evals[visible],
        filtered_weights_r[visible],
        width=0.14,
        alpha=0.9,
        color="C1",
        label="filtered weights" if idx == 0 else None,
    )

    filter_intensity_r = (
        np.abs(trigonometric_krylov_filter(coeffs_r, energy_grid, dt)) ** 2
    )
    filter_intensity_r /= filter_intensity_r.max()
    ax.plot(energy_grid, filter_intensity_r, color="k", linewidth=2)

    ax.axvline(subspace_evals[0], color="C3", linestyle="--", linewidth=1)
    ax.set_ylim(0, 1)
    ax.set_ylabel("weight")
    ax.legend(loc="upper right", title=f"$r={r}$")

axes[-1].set_xlabel("Energy eigenvalue $E_m$")
fig.suptitle(
    r"KQD-learned filter $|f_{\rm KQD}(E)|^2$ sharpening with Krylov dimension $r$",
    y=1.0,
)
fig.tight_layout()
plt.show()

 r    retained    ground overlap    filtered energy
--    --------    --------------    ---------------
 2           2            0.4549           4.184510
 4           4            0.8373           3.310168
 6           6            0.9706           3.158100
 8           7            0.9701           3.158725
10           8            0.9638           3.164503

Subspace ground-state energy (target): 3.136297


<Image src="../docs/images/tutorials/krylov-quantum-diagonalization/extracted-outputs/app-code-sweep-1.avif" alt="Output of the previous code cell" />

## Next steps
If you found this work interesting, you might be interested in the following material:
<Admonition type="tip" title="Recommendations">
- [Sample-based Krylov quantum diagonalization of a fermionic lattice model](/docs/tutorials/sample-based-krylov-quantum-diagonalization)
- [Ground-state energy estimation of the Heisenberg chain with VQE](/docs/tutorials/spin-chain-vqe)
- [Quantum diagonalization algorithms course](/learning/courses/quantum-diagonalization-algorithms/krylov)
</Admonition>


## References

[1] E. N. Epperly, L. Lin, and Y. Nakatsukasa, *A theory of quantum subspace diagonalization*, *SIAM Journal on Matrix Analysis and Applications* **43**, 1263-1290 (2022).

[2] N. Yoshioka, M. Amico, W. Kirby, et al., *Diagonalization of large many-body Hamiltonians on a quantum processor*, [arXiv:2407.14431](https://arxiv.org/abs/2407.14431) (2024).

[3] R. M. Parrish and P. L. McMahon, *Quantum filter diagonalization: quantum eigendecomposition without full quantum phase estimation*, *Physical Review Letters* **122**, 230401 (2019).

[4] G. Lee, S. Choi, J. Huh, and A. F. Izmaylov, *Efficient strategies for reducing sampling error in quantum Krylov subspace diagonalization*, *Digital Discovery* **4**, 954-969 (2025).

[5] G. Lee, M. Kang, J. Hong, S. Fomichev and J. Huh, *Filtered Quantum Phase Estimation*, [arXiv:2510.04294](https://arxiv.org/abs/2510.04294) (2025).